# 01. Wireless Signal Propagation: Model Exploration

This notebook introduces and validates each of the **seven propagation models** implemented in the `propagation` framework:

1. Free Space Path Loss (Friis)
2. Two-Ray Ground Reflection
3. Log-Distance Path Loss
4. Okumura Model
5. Okumura-Hata Model
6. COST-231 Hata Model
7. Log-Normal Shadowing


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure src is discoverable
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from propagation.models import (
    FreeSpacePathLoss, TwoRayGroundReflection, LogDistancePathLoss,
    OkumuraModel, HataModel, COST231HataModel, LogNormalShadowing
)
from propagation.core.units import wavelength

print('Models successfully loaded!')


### 1. Free Space Path Loss (Friis Model)

The Friis equation predicts line-of-sight path loss in an ideal lossless vacuum:

$$PL_{FS}(d) = 20\log_{10}(d) + 20\log_{10}(f) + 20\log_{10}\left(\frac{4\pi}{c}\right)$$


In [ ]:
fspl = FreeSpacePathLoss()
d_km = np.linspace(0.1, 10.0, 100)
f_mhz = 900.0

pl_fspl = fspl.calculate_path_loss(d_km, f_mhz, distance_unit='km', frequency_unit='MHz')

plt.figure(figsize=(7, 4))
plt.plot(d_km, pl_fspl, 'b-', lw=2, label='Friis FSPL (900 MHz)')
plt.xlabel('Distance (km)')
plt.ylabel('Path Loss (dB)')
plt.title('Free Space Path Loss vs Distance')
plt.grid(True, ls=':')
plt.legend()
plt.show()


### 2. Two-Ray Ground Reflection Model

Combines direct LOS ray and single earth specular reflected ray. Asymptotically decays as $d^{-4}$ beyond crossover distance $d_c$.


In [ ]:
two_ray = TwoRayGroundReflection(ht_default=40.0, hr_default=2.0)
d_c = two_ray.crossover_distance(ht=40.0, hr=2.0, frequency_hz=900e6)
print(f'Crossover distance d_c = {d_c:.1f} meters ({d_c/1000:.2f} km)')

pl_exact = two_ray.calculate_path_loss(d_km, f_mhz, distance_unit='km', frequency_unit='MHz', mode='exact')
pl_asymp = two_ray.calculate_path_loss(d_km, f_mhz, distance_unit='km', frequency_unit='MHz', mode='asymptotic')

plt.figure(figsize=(7, 4))
plt.plot(d_km, pl_exact, 'orange', lw=1.8, label='Two-Ray Exact Multipath')
plt.plot(d_km, pl_asymp, 'k--', lw=1.8, label='Two-Ray Asymptotic (d^-4)')
plt.axvline(d_c/1000, color='red', ls=':', label=f'Crossover dc = {d_c/1000:.2f} km')
plt.xlabel('Distance (km)')
plt.ylabel('Path Loss (dB)')
plt.title('Two-Ray Ground Reflection Behavior')
plt.grid(True, ls=':')
plt.legend()
plt.show()


### 3. Log-Distance Path Loss Model

$$PL(d) = PL(d_0) + 10 n \log_{10}(d / d_0)$$


In [ ]:
log_dist_urban = LogDistancePathLoss(environment='urban')
log_dist_indoor = LogDistancePathLoss(environment='indoor_obstructed')

pl_urb = log_dist_urban.calculate_path_loss(d_km, f_mhz, distance_unit='km', frequency_unit='MHz')
print(f'Urban exponent n = {log_dist_urban.n}, Indoor obstructed n = {log_dist_indoor.n}')


### 4-6. Empirical Models: Okumura, Hata, and COST-231

Comparing classical empirical macrocell formulations in an urban environment at 1800 MHz.


In [ ]:
okumura = OkumuraModel(hte_default=30.0, hre_default=1.5, environment='urban')
hata = HataModel(hb_default=30.0, hm_default=1.5, environment='urban_medium')
cost231 = COST231HataModel(hb_default=30.0, hm_default=1.5, environment='medium_city')

pl_oku = okumura.calculate_path_loss(d_km, 1800.0, distance_unit='km', frequency_unit='MHz')
pl_hata = hata.calculate_path_loss(d_km, 1800.0, distance_unit='km', frequency_unit='MHz')
pl_cost = cost231.calculate_path_loss(d_km, 1800.0, distance_unit='km', frequency_unit='MHz')

plt.figure(figsize=(7, 4))
plt.plot(d_km, pl_oku, 'r:', lw=2, label='Okumura (1800 MHz)')
plt.plot(d_km, pl_hata, 'm-', lw=2, label='Okumura-Hata (1800 MHz Extrap.)')
plt.plot(d_km, pl_cost, 'brown', lw=2, ls='--', label='COST-231 Hata (1800 MHz)')
plt.xlabel('Distance (km)')
plt.ylabel('Path Loss (dB)')
plt.title('Empirical Cellular Model Comparison (1800 MHz)')
plt.grid(True, ls=':')
plt.legend()
plt.show()


### 7. Stochastic Log-Normal Shadowing

$$PL(d) = \overline{PL}(d) + X_\sigma, \quad X_\sigma \sim \mathcal{N}(0, \sigma^2)$$


In [ ]:
shadow = LogNormalShadowing(n=3.0, sigma_db=6.0, seed=42)
realizations = shadow.generate_realizations(d_km, 1800.0, num_realizations=5, distance_unit='km', frequency_unit='MHz')
mean_pl, lower_95, upper_95 = shadow.calculate_bounds(d_km, 1800.0, confidence=0.95, distance_unit='km', frequency_unit='MHz')

plt.figure(figsize=(7, 4))
for i in range(len(realizations)):
    plt.plot(d_km, realizations[i], color='gray', alpha=0.4, lw=1.0)
plt.plot(d_km, mean_pl, 'b-', lw=2.2, label='Deterministic Mean')
plt.fill_between(d_km, lower_95, upper_95, color='b', alpha=0.15, label='95% Confidence Bounds (+/- 1.96 sigma)')
plt.xlabel('Distance (km)')
plt.ylabel('Path Loss (dB)')
plt.title('Log-Normal Shadowing Realizations (sigma = 6 dB)')
plt.grid(True, ls=':')
plt.legend()
plt.show()
